In [12]:
import os
import sys

sys.path.insert(0, os.path.abspath("../src"))

## 1. Bype Pair Encoding (BPE)

In [ ]:
from collections import defaultdict

corpus = {
    "low": 5,
    "lower": 2,
    "lowest": 6,
    "newer": 3,
}

vocab = {" ".join(list(mot) + ["</w>"]): freq for mot, freq in corpus.items()}
print("Vocabulaire initial (caracteres seuls) :")
for mot, freq in vocab.items():
    print(f"  '{mot}' (freq={freq})")


def get_pair_counts(vocab):
    pairs = defaultdict(int)
    for mot, freq in vocab.items():
        symboles = mot.split()
        for i in range(len(symboles) - 1):
            pairs[(symboles[i], symboles[i + 1])] += freq
    return pairs


def merge_pair(pair, vocab):
    nouveau_vocab = {}
    bigram = " ".join(pair)
    remplacement = "".join(pair)
    for mot in vocab:
        nouveau_mot = mot.replace(bigram, remplacement)
        nouveau_vocab[nouveau_mot] = vocab[mot]
    return nouveau_vocab


print()
print("=== Fusions successives ===")
for etape in range(5):
    paires = get_pair_counts(vocab)
    meilleure_paire = max(paires, key=paires.get)
    print(
        f"Etape {etape+1} : fusion de {meilleure_paire} (vue {paires[meilleure_paire]} \
            fois) -> '{''.join(meilleure_paire)}'"
    )
    vocab = merge_pair(meilleure_paire, vocab)

print()
print("Vocabulaire apres 5 fusions :")
for mot, freq in vocab.items():
    print(f"  '{mot}' (freq={freq})")

Vocabulaire initial (caracteres seuls) :
  'l o w </w>' (freq=5)
  'l o w e r </w>' (freq=2)
  'l o w e s t </w>' (freq=6)
  'n e w e r </w>' (freq=3)

=== Fusions successives ===
Etape 1 : fusion de ('l', 'o') (vue 13 fois) -> 'lo'
Etape 2 : fusion de ('lo', 'w') (vue 13 fois) -> 'low'
Etape 3 : fusion de ('low', 'e') (vue 8 fois) -> 'lowe'
Etape 4 : fusion de ('lowe', 's') (vue 6 fois) -> 'lowes'
Etape 5 : fusion de ('lowes', 't') (vue 6 fois) -> 'lowest'

Vocabulaire apres 5 fusions :
  'low </w>' (freq=5)
  'lowe r </w>' (freq=2)
  'lowest </w>' (freq=6)
  'n e w e r </w>' (freq=3)


In [16]:
from tokenization.bpe import tokenize_with_bpe, train_bpe_tokenizer

corpus = [
    "the delivery was fast",
    "the delivery was slow",
    "the shipping was fast",
    "the shipping was slow",
    "the product quality was great",
    "the product quality was poor",
    "customer service was helpful",
    "customer service was rude",
    "the packaging was damaged",
    "the packaging was excellent",
] * 20

tokenizer = train_bpe_tokenizer(corpus, vocab_size=100, min_frequency=2)
print("\nTaille du vocabulaire appris :", tokenizer.get_vocab_size())





Taille du vocabulaire appris : 100


In [ ]:
# --- BLOC 3 : mot connu vs mot jamais vu vs mot totalement etranger ---
print("'delivery' (mot connu) :", tokenize_with_bpe(tokenizer, "delivery"))
print(
    "'deliveryman' (jamais vu, fragment connu) :",
    tokenize_with_bpe(tokenizer, "deliveryman"),
)
print("'xyzabc123' (aucun rapport) :", tokenize_with_bpe(tokenizer, "xyzabc123"))


# --- BLOC 4 : effet de vocab_size sur le niveau de decoupe ---
for taille in [20, 50, 200]:
    t = train_bpe_tokenizer(corpus, vocab_size=taille, min_frequency=2)
    tokens = tokenize_with_bpe(t, "delivery")
    print(
        f"vocab_size={taille:4} -> 'delivery' decoupe en : {tokens} (vocabulaire reel \
            : {t.get_vocab_size()})"
    )

'delivery' (mot connu) : ['delivery']
'deliveryman' (jamais vu, fragment connu) : ['delivery', 'm', 'a', 'n']
'xyzabc123' (aucun rapport) : ['x', 'y', '[UNK]', 'a', '[UNK]', 'c', '[UNK]', '[UNK]', '[UNK]']



vocab_size=  20 -> 'delivery' decoupe en : ['d', 'e', 'l', 'i', 'v', 'e', 'r', 'y'] (vocabulaire reel : 24)



vocab_size=  50 -> 'delivery' decoupe en : ['de', 'li', 'v', 'er', 'y'] (vocabulaire reel : 50)



vocab_size= 200 -> 'delivery' decoupe en : ['delivery'] (vocabulaire reel : 101)


## . Wordpiece

In [ ]:
corpus_jouet = {"low": 5, "lower": 2, "lowest": 6, "newer": 3}
vocab = {" ".join(list(mot) + ["</w>"]): freq for mot, freq in corpus_jouet.items()}


def get_symbol_freqs(vocab):
    freqs = defaultdict(int)
    for mot, freq in vocab.items():
        for symbole in mot.split():
            freqs[symbole] += freq
    return freqs


def get_pair_counts(vocab):
    pairs = defaultdict(int)
    for mot, freq in vocab.items():
        symboles = mot.split()
        for i in range(len(symboles) - 1):
            pairs[(symboles[i], symboles[i + 1])] += freq
    return pairs


pair_counts = get_pair_counts(vocab)
symbol_freqs = get_symbol_freqs(vocab)

resultats = []
for paire, freq_paire in pair_counts.items():
    a, b = paire
    score_bpe = freq_paire
    score_wp = freq_paire / (symbol_freqs[a] * symbol_freqs[b])
    resultats.append((paire, freq_paire, score_bpe, score_wp))

print("--- Top 3 selon BPE (frequence brute) ---")
for paire, fp, s_bpe, s_wp in sorted(resultats, key=lambda x: -x[2])[:3]:
    print(f"  {paire} : freq={fp}, score_BPE={s_bpe}, score_WordPiece={s_wp:.4f}")

print("\n--- Top 3 selon WordPiece (score normalise) ---")
for paire, fp, s_bpe, s_wp in sorted(resultats, key=lambda x: -x[3])[:3]:
    print(f"  {paire} : freq={fp}, score_BPE={s_bpe}, score_WordPiece={s_wp:.4f}")

In [ ]:
# --- BLOC 2 : version bibliotheque -- BPE vs WordPiece cote a cote ---

from tokenization.bpe import tokenize_with_bpe, train_bpe_tokenizer
from tokenization.wordpiece import tokenize_with_wordpiece, train_wordpiece_tokenizer

corpus = [
    "the delivery was fast",
    "the delivery was slow",
    "the shipping was fast",
    "the shipping was slow",
    "the product quality was great",
    "the product quality was poor",
    "customer service was helpful",
    "customer service was rude",
    "the packaging was damaged",
    "the packaging was excellent",
] * 20

tok_bpe = train_bpe_tokenizer(corpus, vocab_size=60, min_frequency=2)
tok_wp = train_wordpiece_tokenizer(corpus, vocab_size=60, min_frequency=2)


# --- BLOC 3 : comparaison sur plusieurs mots, avec attention au prefixe "##" ---
mots_test = ["delivery", "deliveryman", "packaging", "unpackaged"]

for mot in mots_test:
    print(f"'{mot}':")
    print("  BPE      :", tokenize_with_bpe(tok_bpe, mot))
    print("  WordPiece:", tokenize_with_wordpiece(tok_wp, mot))